In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import (
    train_test_split, StratifiedKFold, GridSearchCV, cross_val_predict
)
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    mean_squared_error, accuracy_score,
    precision_score, recall_score, f1_score
)

from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier

In [2]:
def compute_metrics(y_true, y_pred, label=""):
    mse  = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average="weighted", zero_division=0)
    rec  = recall_score(y_true, y_pred, average="weighted", zero_division=0)
    f1   = f1_score(y_true, y_pred, average="weighted", zero_division=0)
    return {
        "Set": label,
        "MSE": round(mse, 4),
        "RMSE": round(rmse, 4),
        "Accuracy": round(acc, 4),
        "Precision": round(prec, 4),
        "Recall": round(rec, 4),
        "F1": round(f1, 4)
    }

In [7]:
df = sns.load_dataset('titanic')
df = df.drop(columns=['deck', 'embark_town', 'alive', 'class', 'who', 'adult_male', 'sibsp', 'parch', 'alone', 'embarked'])

df['age'] = df['age'].fillna(df['age'].median())
df['sex'] = df['sex'].map({'male': 0, 'female': 1})

Q1 = df['fare'].quantile(0.25)
Q3 = df['fare'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

df['fare'] = np.where(df['fare'] > upper_bound, upper_bound, 
             np.where(df['fare'] < lower_bound, lower_bound, df['fare']))
df = df.rename(columns={'survived': 'target'})

df_final = df.iloc[:, list(range(1, df.shape[1])) + [0]]

X_train, X_test = train_test_split(df_final, test_size=0.2, random_state=42)

scaler = MinMaxScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=df_final.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=df_final.columns)
X_train = X_train_scaled.iloc[:, :-1]
X_test = X_test_scaled.iloc[:, :-1]
y_train = X_train_scaled.iloc[:, -1]
y_test = X_test_scaled.iloc[:, -1]

print(X_train_scaled.head())

   pclass  sex       age      fare  target
0     0.0  0.0  0.566474  0.434224     0.0
1     0.5  0.0  0.283740  0.198067     0.0
2     1.0  0.0  0.396833  0.120745     0.0
3     1.0  0.0  0.321438  0.119666     0.0
4     1.0  1.0  0.070118  0.476503     0.0


In [9]:
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

models = {
    "Naive Bayes": {
        "pipeline": Pipeline([
            ("scaler", MinMaxScaler()),
            ("clf", GaussianNB())
        ]),
        "params": {
            "clf__var_smoothing": np.logspace(-12, -6, 7)
        }
    },

    "Logistic Regression": {
        "pipeline": Pipeline([
            ("scaler", MinMaxScaler()),
            ("clf", LogisticRegression(max_iter=1000, random_state=42))
        ]),
        "params": {
            "clf__C": [0.001, 0.01, 0.1, 1, 10, 100],
            "clf__solver": ["lbfgs", "liblinear"],
            "clf__penalty": ["l2"]
        }
    },

    "Random Forest": {
        "pipeline": Pipeline([
            ("scaler", MinMaxScaler()),
            ("clf", RandomForestClassifier(random_state=42))
        ]),
        "params": {
            "clf__n_estimators": [50, 100, 200],
            "clf__max_depth": [None, 5, 10],
            "clf__min_samples_split": [2, 5],
            "clf__min_samples_leaf": [1, 2]
        }
    },

    "Neural Network": {
        "pipeline": Pipeline([
            ("scaler", MinMaxScaler()),
            ("clf", MLPClassifier(max_iter=500, random_state=42))
        ]),
        "params": {
            "clf__hidden_layer_sizes": [(64,), (128,), (64, 32), (128, 64)],
            "clf__activation": ["relu", "tanh"],
            "clf__alpha": [0.0001, 0.001, 0.01],
            "clf__learning_rate_init": [0.001, 0.01]
        }
    }
}

In [11]:
all_results = []

for name, config in models.items():
    print(f"\n{'='*60}")
    print(f"  Model: {name}")
    print(f"{'='*60}")

    gs = GridSearchCV(
        estimator=config["pipeline"],
        param_grid=config["params"],
        cv=cv,
        scoring="f1_weighted",
        n_jobs=-1,
        verbose=0
    )
    gs.fit(X_train, y_train)

    best_model = gs.best_estimator_
    print(f"  The best params : {gs.best_params_}")
    print(f"  CV F1 (val. set)   : {gs.best_score_:.4f}")

    y_train_pred = cross_val_predict(best_model, X_train, y_train, cv=cv)
    train_metrics = compute_metrics(y_train, y_train_pred, label="Train (CV)")

    y_test_pred = best_model.predict(X_test)
    test_metrics = compute_metrics(y_test, y_test_pred, label="Test")

    for m in [train_metrics, test_metrics]:
        print(f"\n  [{m['Set']}]")
        for k, v in m.items():
            if k != "Set":
                print(f"    {k:<12}: {v}")

    all_results.append({"Model": name, **train_metrics})
    all_results.append({"Model": name, **test_metrics})


  Model: Naive Bayes
  The best params : {'clf__var_smoothing': 1e-12}
  CV F1 (val. set)   : 0.7633

  [Train (CV)]
    MSE         : 0.2388
    RMSE        : 0.4886
    Accuracy    : 0.7612
    Precision   : 0.7688
    Recall      : 0.7612
    F1          : 0.7635

  [Test]
    MSE         : 0.257
    RMSE        : 0.5069
    Accuracy    : 0.743
    Precision   : 0.7534
    Recall      : 0.743
    F1          : 0.7449

  Model: Logistic Regression
  The best params : {'clf__C': 10, 'clf__penalty': 'l2', 'clf__solver': 'lbfgs'}
  CV F1 (val. set)   : 0.7884

  [Train (CV)]
    MSE         : 0.2093
    RMSE        : 0.4575
    Accuracy    : 0.7907
    Precision   : 0.7884
    Recall      : 0.7907
    F1          : 0.7887

  [Test]
    MSE         : 0.1955
    RMSE        : 0.4422
    Accuracy    : 0.8045
    Precision   : 0.8036
    Recall      : 0.8045
    F1          : 0.8028

  Model: Random Forest
  The best params : {'clf__max_depth': 5, 'clf__min_samples_leaf': 1, 'clf__min_samp

In [12]:
print(f"\n\n{'='*60}")
print("Metrics")
print(f"{'='*60}")
df_results = pd.DataFrame(all_results)
df_results = df_results.set_index(["Model", "Set"])
print(df_results.to_string())



Metrics
                                   MSE    RMSE  Accuracy  Precision  Recall      F1
Model               Set                                                            
Naive Bayes         Train (CV)  0.2388  0.4886    0.7612     0.7688  0.7612  0.7635
                    Test        0.2570  0.5069    0.7430     0.7534  0.7430  0.7449
Logistic Regression Train (CV)  0.2093  0.4575    0.7907     0.7884  0.7907  0.7887
                    Test        0.1955  0.4422    0.8045     0.8036  0.8045  0.8028
Random Forest       Train (CV)  0.1685  0.4105    0.8315     0.8339  0.8315  0.8264
                    Test        0.1899  0.4358    0.8101     0.8128  0.8101  0.8061
Neural Network      Train (CV)  0.1770  0.4207    0.8230     0.8240  0.8230  0.8182
                    Test        0.1788  0.4228    0.8212     0.8266  0.8212  0.8167
